In [ ]:
#| default_exp core

In [7]:
#| hide
import nbdev; nbdev.nbdev_export()

In [8]:
#| export
import torch
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True
import regex as re
import pandas as pd

In [9]:
#| export
import os
from os import environ

def init_instance():
    os.environ["USE_DEEPSPEED"] = "1"
    os.environ["MASTER_PORT"]=str(6000+int(environ.get('INSTANCE', '0')))
    model_path = environ.get('MODEL', 'poetry')
    flavor_id = model_path + environ.get('CUDA_VISIBLE_DEVICES', '0') + environ.get('INSTANCE', '0')
    from tendo import singleton
    me = singleton.SingleInstance(flavor_id=flavor_id)
    return me, model_path

In [10]:
#| export
from rest.storage import logs, connection
from sqlalchemy.dialects.postgresql import insert

def log(request, type, hash, log):
    model = environ.get('MODEL', 'poetry')
    sessionid = request.cookies.get('sessionid')
    insert_stmt = insert(logs).values(ip=request.headers.get('X-Real-IP'), origin=request.headers.get('Origin'),
                                      agent=request.headers.get('User-Agent'), fs=request.headers.get('X-Forwarded-Server'),
                                      ff=request.headers.get('X-Forwarded-For'),
                                      session=f'{sessionid}', type=type, hash=hash, 
                                      log=str(log), model=model)
    connection.execute(insert_stmt)

In [11]:
#| export
def get_ban_ip(ip):
    ban_type = pd.read_sql_query("select ban_type from ban where ip = %(ip)s", connection, params={"ip":ip})
    if ban_type.empty: return 0
    return int(ban_type.iloc[0]['ban_type'])

def get_ban(request):
    ip = request.headers.get('X-Real-IP')
    return get_ban_ip(ip)

In [12]:
get_ban_ip('100.21.134.76')

0

In [13]:
#| export
import regex as re

def fix_string(string) -> str:
    in_word = string
    in_between_words = ['-', '–']
    in_sentences = ['«', '(', '[', '{', '"', '„', '\'']

    for item in in_between_words:
        regex = r'\w[%s]\s\w' % item
        in_word = re.findall(regex, string)

        for x in in_word:
            a = x[:1]; b = x[3:4]
            string = string.replace(x, a + '-' + b)

    for item in in_sentences:
        string = string.replace(f' {item} ', f' {item}')

    return string

def process_seq(generated_sequences):
    reg_text = [re.match(r'[\w\W]*[\.!?]\n', item) for item in generated_sequences]
    reg_text2 = [re.match(r'[\w\W]*[\.!?]', item) for item in generated_sequences]
    result = [reg_item[0] if reg_item else reg_item2[0] if reg_item2 else item for reg_item, reg_item2, item in zip(reg_text, reg_text2, generated_sequences)]
    result = [fix_string(s) for s in result]
    return result 

In [14]:
#| export
def bad_points(tokenizer, point):
    result = []
    for i in range(2, 50):
        code = tokenizer.encode(point*i)
        if len(code) == 1:
            result += [code]
    return result

def bad_words(tokenizer, allow_linebreak):
    bad_symbols = ['[','(','\xa0','*','­', '~', '_', '\\', '\n\n', '\uf04a', '\ufeff', '\u2028']
    bad_words_ids = [tokenizer.encode(s) for s in bad_symbols]
    
    eot = tokenizer.encode('a<|endoftext|>')[1]
    if eot: bad_words_ids += [[eot]]
    
    for point in ['.','*','_','-','\xa0','!']:
        bad_words_ids += bad_points(tokenizer, point)
    linebreaks = [tokenizer.encode(s) for s in ['\n', ' \n']]
    bad_words_ids += [] if allow_linebreak else linebreaks
    return bad_words_ids

In [15]:
#| export
def generate(model, tokenizer, seq_length, prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length= length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            no_repeat_ngram_size=2,
            bad_words_ids = bad_words(tokenizer, allow_linebreak)
        )
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)